# Customer Retention & RFM Analysis — Data Cleaning

- This notebook cleans the **Online Retail II** dataset (UCI Machine Learning Repository) to prepare it for customer segmentation (RFM) analysis.

- Dataset source: https://archive.ics.uci.edu/dataset/502/online+retail+ii  
- Coverage: UK-based online gift retailer, transactions from Dec 2009 – Dec 2011

## 1. Load Data & Initial Inspection

In [6]:
import numpy as np
import pandas as pd

df = pd.read_excel('/Users/piyushkalra/Downloads/online_retail_II.xlsx', engine='openpyxl')

In [7]:
print(df.head())
print(df.info())

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      52

## 2. Identify Cancelled Orders and Data Anomalies

Invoice numbers starting with 'C' indicate cancelled orders, and 'A' indicates accounting adjustments. Before cleaning, I checked how prevalent these were, along with negative quantities and prices.

In [8]:
print("Negative quantities:", (df['Quantity'] < 0).sum())
print("Negative prices:", (df['Price'] < 0).sum())
print(df['Invoice'].astype(str).str[0].value_counts())

Negative quantities: 12326
Negative prices: 3
Invoice
5    406763
4    108489
C     10206
A         3
Name: count, dtype: int64


## 3. Remove Cancelled and Adjustment Orders

Cancelled ('C') and adjustment ('A') invoices don't represent genuine completed purchases, so they're excluded before further analysis.

In [9]:
df_clean = df[~df['Invoice'].astype(str).str.startswith(('C', 'A'))]

print("Original rows:", len(df))
print("After removing C/A:", len(df_clean))
print("Remaining negative quantities:", (df_clean['Quantity'] < 0).sum())

Original rows: 525461
After removing C/A: 515252
Remaining negative quantities: 2121


## 4. Investigate Remaining Negative Quantities

Even after removing cancelled orders, some negative quantity rows were still showing up. Instead of just dropping them, I wanted to actually look at what they were before deciding what to do.

In [10]:
negative_sample = df_clean[df_clean['Quantity'] < 0]
print(negative_sample[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']].head(10))

print("Missing Customer ID among negative quantities:", negative_sample['Customer ID'].isnull().sum())
print("Total negative quantity rows:", len(negative_sample))

     Invoice StockCode      Description  Quantity  Price  Customer ID
263   489464     21733     85123a mixed       -96    0.0          NaN
283   489463     71477            short      -240    0.0          NaN
284   489467    85123A      21733 mixed      -192    0.0          NaN
470   489521     21646              NaN       -50    0.0          NaN
3114  489655     20683              NaN       -44    0.0          NaN
3162  489660     35956             lost     -1043    0.0          NaN
3168  489663    35605A          damages      -117    0.0          NaN
4296  489806     18010              NaN      -770    0.0          NaN
4538  489820     21133  invcd as 84879?      -720    0.0          NaN
4566  489821    85049G              NaN      -240    0.0          NaN
Missing Customer ID among negative quantities: 2121
Total negative quantity rows: 2121


 These rows had no Customer ID and descriptions like "damages," "lost," and "mixed" indicating internal adjustments, not customer transactions. So i removed these rows as they were of no use.

## 5. Remove Rows with Missing Customer ID

In [11]:
df_final = df_clean.dropna(subset=['Customer ID'])

print("Rows after removing missing Customer ID:", len(df_final))
print("Remaining negative quantities:", (df_final['Quantity'] < 0).sum())

Rows after removing missing Customer ID: 407695
Remaining negative quantities: 0


## 6. Remove Duplicate Rows and Fix Data Types

Removed all the duplicate rows in the data as they are just repeated entries and converting Customer ID from float datatype to Str, as Customer ID is not a number but a label

In [14]:
print("Missing descriptions:", df_final['Description'].isnull().sum())
print("Duplicate rows:", df_final.duplicated().sum())

print("Before removing duplicates:", len(df_final))
df_final = df_final.drop_duplicates()
print("After removing duplicates:", len(df_final))

df_final['Customer ID'] = df_final['Customer ID'].astype(int).astype(str)
print("Datatype of Customer ID:-",df_final['Customer ID'].dtype)

Missing descriptions: 0
Duplicate rows: 0
Before removing duplicates: 400947
After removing duplicates: 400947
Datatype of Customer ID:- str


## 7. Calculating order value for later Monetary calculations

In [15]:
df_final['order_value'] = df_final['Quantity'] * df_final['Price']

## 8. Investigate Outliers in Quantity and Price

Extreme values in Quantity and Price could indicate either genuine wholesale transactions or data errors. I checked the actual rows behind these outliers before deciding how to handle them because it can either be an wholesale order or just another "Adjustment" entries.

In [16]:
print(df_final['Quantity'].describe())
print(df_final['Price'].describe())

print(df_final.nlargest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']])
print(df_final.nlargest(10, 'Price')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Customer ID']])

count    400947.000000
mean         13.768523
std          97.639816
min           1.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       19152.000000
Name: Quantity, dtype: float64
count    400947.000000
mean          3.305571
std          35.046376
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max       10953.500000
Name: Price, dtype: float64
       Invoice StockCode                         Description  Quantity  Price  \
90857   497946     37410  BLACK AND WHITE PAISLEY FLOWER MUG     19152   0.10   
127166  501534     21099         SET/6 STRAWBERRY PAPER CUPS     12960   0.10   
127168  501534     21091         SET/6 WOODLAND PAPER PLATES     12960   0.10   
127169  501534     21085           SET/6 WOODLAND PAPER CUPS     12744   0.10   
127167  501534     21092       SET/6 STRAWBERRY PAPER PLATES     12480   0.10   
135027  502269     21984    PACK OF 12 PINK PAISLEY TISSUES      10000   0.25   
135028

- High quantity outliers (up to 19,152 units) are genuine wholesale bulk orders, consistent with this dataset's documented customer base of wholesalers. These are retained.
- High price outliers (up to £10,953.50) are all associated with StockCode 'M' ("Manual")  internal adjustment entries, not real product sales. These are removed.

In [17]:
print("Manual entries found:", (df_final['StockCode'] == 'M').sum())
df_final = df_final[df_final['StockCode'] != 'M']
print("Shape after removing Manual entries:", df_final.shape)

print(df_final['Price'].describe())

Manual entries found: 421
Shape after removing Manual entries: (400526, 9)
count    400526.000000
mean          3.067910
std           5.133548
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max         850.000000
Name: Price, dtype: float64


## 8. Save Cleaned Data

The final cleaned dataset is saved for use in subsequent SQL and RFM analysis notebooks.

In [18]:
df_final.to_csv('/Users/piyushkalra/Desktop/cleaned_retail_data.csv', index=False)
print("Final cleaned shape:", df_final.shape)

Final cleaned shape: (400526, 9)
